# Databricks/PySpark for Data Analysts/Engineers — Module 3: Transformations and Aggregations

This module extends the DataFrame API from Module 2 with aggregations (`groupBy`), the
`pyspark.sql.functions` built-in function library, conditional expressions
(`when`/`otherwise`, the equivalent of SQL's `CASE WHEN`), and handling `NULL` values.

## Table of contents
1. [groupBy and aggregations](#sec1)
2. [Multiple aggregations at once](#sec2)
3. [Built-in functions: strings and dates](#sec3)
4. [Conditional expressions: when/otherwise](#sec4)
5. [Handling NULL values](#sec5)
6. [Summary and exercises](#sec6)


We start (or grab the already-running) `SparkSession` -- the entry point to everything in Spark. In Databricks, `spark` is already available automatically in every notebook; the same code also works locally.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("spark_course").getOrCreate()
spark


Let's load our shared dataset -- online store data (`customers`, `products`, `employees`, `orders`, `order_items`), the exact same schema as the SQL course, but this time as Spark DataFrames.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)

# --- customers -----------------------------------------------------------
n_customers = 300
cities = ["Warsaw", "Krakow", "Gdansk", "Wroclaw", "Poznan"]
segments = ["New", "Standard", "Premium"]
segment_probs = [0.35, 0.45, 0.20]

customers_pd = pd.DataFrame({
    "customer_id": np.arange(1, n_customers + 1),
    "city": rng.choice(cities, size=n_customers, p=[0.30, 0.25, 0.15, 0.15, 0.15]),
    "segment": rng.choice(segments, size=n_customers, p=segment_probs),
    "days_since_signup": rng.integers(1, 900, size=n_customers),
})

# --- products ------------------------------------------------------------
n_products = 40
categories = ["Electronics", "Clothing", "Home", "Sports", "Books"]
avg_prices = {"Electronics": 1200, "Clothing": 150, "Home": 250, "Sports": 300, "Books": 50}

product_categories = rng.choice(categories, size=n_products)
products_pd = pd.DataFrame({
    "product_id": np.arange(1, n_products + 1),
    "category": product_categories,
    "price": [
        round(max(5.0, rng.normal(avg_prices[c], avg_prices[c] * 0.3)), 2)
        for c in product_categories
    ],
})

# --- employees ----------------------------------------------------------
n_employees = 12
regions = ["North", "South", "East", "West"]
employees_pd = pd.DataFrame({
    "employee_id": np.arange(1, n_employees + 1),
    "region": rng.choice(regions, size=n_employees),
    "department": rng.choice(["Sales", "Support"], size=n_employees, p=[0.6, 0.4]),
})

# --- orders ------------------------------------------------------------
n_orders = 3000
order_customer_id = rng.choice(customers_pd["customer_id"], size=n_orders)
order_employee_id = rng.choice(employees_pd["employee_id"], size=n_orders)

segment_map = customers_pd.set_index("customer_id")["segment"]
order_segment = pd.Series(order_customer_id).map(segment_map).values

# cancellation probability depends on customer segment (deliberately built in)
cancel_prob = np.where(
    order_segment == "New", 0.22,
    np.where(order_segment == "Standard", 0.10, 0.04),
)
cancelled = rng.binomial(1, cancel_prob)
order_status = np.where(
    cancelled == 1, "Cancelled",
    rng.choice(["Placed", "Shipped", "Delivered"], size=n_orders, p=[0.2, 0.3, 0.5]),
)

base_date = pd.Timestamp("2024-01-01")
day_offset = rng.integers(0, 730, size=n_orders)

orders_pd = pd.DataFrame({
    "order_id": np.arange(1, n_orders + 1),
    "customer_id": order_customer_id,
    "employee_id": order_employee_id,
    "order_date": [base_date + pd.Timedelta(days=int(d)) for d in day_offset],
    "status": order_status,
})

# --- order_items ------------------------------------------------------
item_rows = []
item_id = 1
product_price = products_pd.set_index("product_id")["price"]

for order_id in orders_pd["order_id"]:
    n_items = rng.integers(1, 5)
    chosen_products = rng.choice(products_pd["product_id"], size=n_items, replace=False)
    for product_id in chosen_products:
        quantity = int(rng.integers(1, 4))
        discount = rng.choice([0.0, 0.05, 0.10, 0.15], p=[0.55, 0.20, 0.15, 0.10])
        item_rows.append({
            "item_id": item_id,
            "order_id": int(order_id),
            "product_id": int(product_id),
            "quantity": quantity,
            "unit_price": float(product_price[product_id]),
            "discount": float(discount),
        })
        item_id += 1

order_items_pd = pd.DataFrame(item_rows)

# --- convert to Spark DataFrames ---------------------------------------
customers = spark.createDataFrame(customers_pd)
products = spark.createDataFrame(products_pd)
employees = spark.createDataFrame(employees_pd)
orders = spark.createDataFrame(orders_pd)
order_items = spark.createDataFrame(order_items_pd)

customers.show(5)


<a id="sec1"></a>
## 1. groupBy and aggregations

`groupBy()` + `agg()` (or shorthand methods like `.count()`) correspond to SQL's `GROUP
BY` -- exactly what we covered in Module 4 of the SQL course, now in Spark.

In [ ]:
customers.groupBy("segment").count().show()


In [ ]:
from pyspark.sql import functions as F

products.groupBy("category").agg(F.avg("price").alias("avg_price")).show()


> ⚡ **groupBy() alone computes nothing**
>
> Like the other transformations from Module 1 -- `df.groupBy("x")` by itself returns a `GroupedData` object, not a result yet. Only chaining `.agg(...)` (or `.count()`) and calling an action (`.show()`) actually triggers computation.

<a id="sec2"></a>
## 2. Multiple aggregations at once

`agg()` accepts multiple aggregate functions at once -- exactly like multiple columns
after `SELECT` in a `GROUP BY` query.

In [ ]:
products.groupBy("category").agg(
    F.count("*").alias("product_count"),
    F.avg("price").alias("avg_price"),
    F.min("price").alias("min_price"),
    F.max("price").alias("max_price"),
    F.round(F.stddev("price"), 2).alias("stddev_price"),
).orderBy(F.col("avg_price").desc()).show()


We can also aggregate over MULTIPLE columns at once -- Spark groups by every unique
COMBINATION of values in those columns, exactly like `GROUP BY column1, column2` in
SQL.

In [ ]:
orders_with_customers = orders.join(customers, "customer_id")

orders_with_customers.groupBy("segment", "status").agg(
    F.count("*").alias("order_count")
).orderBy("segment", "status").show(20)


<a id="sec3"></a>
## 3. Built-in functions: strings and dates

`pyspark.sql.functions` (usually imported as `F`) is a huge library of ready-made
functions -- string, math, date, aggregate, window (Module 7), and more. Instead of
writing your own logic in Python (which would be slow -- see Module 8 on UDFs), it's
almost always worth checking first whether the function you need already exists
there.

In [ ]:
products.select(
    "category",
    F.upper("category").alias("category_upper"),
    F.length("category").alias("name_length"),
).show(5)


In [ ]:
orders.select(
    "order_date",
    F.year("order_date").alias("year"),
    F.month("order_date").alias("month"),
    F.date_add("order_date", 30).alias("date_plus_30_days"),
).show(5)


> ⚡ **Cheat sheet: where to look for functions**
>
> The full list is in the official `pyspark.sql.functions` documentation -- worth keeping handy. The most commonly used groups: string (`upper`, `lower`, `trim`, `substring`, `concat`, `regexp_replace`), date (`year`, `month`, `datediff`, `date_add`), math (`round`, `abs`, `sqrt`), and conditional (`when`, `coalesce`, `isnull`).

<a id="sec4"></a>
## 4. Conditional expressions: when/otherwise

`F.when(condition, value).when(...).otherwise(default)` is the equivalent of SQL's `CASE
WHEN ... THEN ... ELSE ... END` -- we build a new column based on a series of conditions
checked in order.

In [ ]:
customers_with_type = customers.withColumn(
    "customer_type",
    F.when(F.col("segment") == "Premium", "VIP")
    .when(F.col("segment") == "New", "Newcomer")
    .otherwise("Regular"),
)
customers_with_type.groupBy("customer_type").count().show()


> ⚠️ **The order of conditions in when() matters**
>
> Just like SQL's `CASE WHEN`, Spark checks conditions IN ORDER and takes the FIRST match -- if two conditions could both match the same row, the earlier one in the `.when()` chain wins.

<a id="sec5"></a>
## 5. Handling NULL values

Our data happens to have no gaps, but in practice you'll run into them almost always.
The key tools: `isNull()`/`isNotNull()` (filtering), `fillna()`/`na.fill()` (filling in),
`dropna()`/`na.drop()` (removing rows with gaps), `F.coalesce()` (the first non-null
value out of several columns).

In [ ]:
# Small test data with gaps, to show the mechanics
test_data = spark.createDataFrame(
    [(1, "Warsaw"), (2, None), (3, "Gdansk")], ["id", "city"]
)
test_data.filter(F.col("city").isNull()).show()


In [ ]:
test_data.fillna({"city": "Unknown"}).show()


In [ ]:
test_data.na.drop().show()


> ⚠️ **== None doesn't work the way it does in Python**
>
> `F.col("city") == None` will NOT catch gaps (in SQL/Spark logic, comparing anything with `NULL` always gives `NULL`, never `True`). ALWAYS use `.isNull()` / `.isNotNull()` to check for gaps, never `== None`.

<a id="sec6"></a>
## 6. Summary and exercises

In this module we covered:
- `groupBy()` + `agg()` with multiple aggregate functions at once,
- grouping by multiple columns simultaneously,
- the `pyspark.sql.functions` built-in function library (strings, dates),
- conditional expressions `when()`/`otherwise()` (the equivalent of `CASE WHEN`),
- handling `NULL` values: `isNull`/`isNotNull`, `fillna`, `dropna`, `coalesce`.

### 📝 Exercise 1: Average order value per city

Join `orders` with `customers`, group by `city`, and count the number of orders in each city.

<details>
<summary>Show solution</summary>

```python
orders.join(customers, "customer_id").groupBy("city").agg(
    F.count("*").alias("order_count")
).orderBy(F.col("order_count").desc()).show()
```

</details>

### 📝 Exercise 2: Categorizing products by price

Add a column `price_tier` to `products` using `when/otherwise`: `'Cheap'` for price below 100, `'Mid'` for 100-500, `'Expensive'` above 500.

<details>
<summary>Show solution</summary>

```python
products.withColumn(
    "price_tier",
    F.when(F.col("price") < 100, "Cheap")
    .when(F.col("price") <= 500, "Mid")
    .otherwise("Expensive"),
).groupBy("price_tier").count().show()
```

</details>

### 📝 Exercise 3: The month with the most orders

Extract the month (`F.month`) from `order_date`, group by it, and find the month with the highest order count.

<details>
<summary>Show solution</summary>

```python
orders.withColumn("month", F.month("order_date")).groupBy("month").agg(
    F.count("*").alias("order_count")
).orderBy(F.col("order_count").desc()).show(3)
```

</details>

### 📝 Exercise 4: Filling in gaps

For `test_data` from section 5, fill the gaps in `city` with the value `'No data'` and print the result.

<details>
<summary>Show solution</summary>

```python
test_data.fillna({"city": "No data"}).show()
```

</details>

> 🔥 **Challenge: a full breakdown of order value by segment and status**
>
> Join `orders`, `customers`, and `order_items`, add an `item_value` column (as in Module 2), aggregate the SUM of value per `(segment, status)`, and add a column `value_tier` via `when/otherwise`: `'High'` when the sum exceeds 5000, otherwise `'Standard'`.

<details>
<summary>Show solution</summary>

```python
joined = (
    orders.join(customers, "customer_id")
    .join(order_items, "order_id")
    .withColumn("item_value", F.col("quantity") * F.col("unit_price") * (1 - F.col("discount")))
)

result = joined.groupBy("segment", "status").agg(
    F.round(F.sum("item_value"), 2).alias("total_value")
).withColumn(
    "value_tier",
    F.when(F.col("total_value") > 5000, "High").otherwise("Standard"),
)
result.orderBy(F.col("total_value").desc()).show(20)
```

</details>

## What's next?

In Module 3 we joined tables "on the fly" before formally covering `join`. **Module 4**
does it properly: join types (inner/left/right/outer/semi/anti), broadcast joins, and a
first look at how expensive joining large tables can get.